In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType
from pyspark.sql import Row
from datetime import datetime

# Define schema
employees_schema = StructType([
    StructField("EmployeeID", StringType(), False),
    StructField("HireDate", DateType(), False),
    StructField("Department", StringType(), True),
    StructField("Region", StringType(), True),
    StructField("JobTitle", StringType(), False),
    StructField("Category", StringType(), False),
    StructField("Salary", DoubleType(), False),
    StructField("YearsExperience", IntegerType(), False)
])

# Sample data
employees_data = [
    ("EMP001", datetime(2020, 3, 15), "West",  "Engineering",  "Data Engineer",    "Full-Time", 95000.00, 4),
    ("EMP002", datetime(2019, 7, 22), "East",  "Marketing",    "Analyst",          "Full-Time", 72000.00, 6),
    ("EMP003", datetime(2021, 1, 10), "South", "HR",           "HR Specialist",    "Part-Time", 55000.00, 2),
    ("EMP004", datetime(2018, 5, 30), "North", "Engineering",  "Senior Dev",       "Full-Time", 120000.00, 8),
    ("EMP005", datetime(2022, 9, 5),  "West",  "Finance",      "Accountant",       "Full-Time", 80000.00, 3),
    ("EMP006", datetime(2020, 11, 18),"East",  "Engineering",  "ML Engineer",      "Full-Time", 110000.00, 5),
    ("EMP007", datetime(2017, 4, 12), "South", "Sales",        "Sales Manager",    "Full-Time", 90000.00, 9),
    ("EMP008", datetime(2023, 2, 28), "North", "Marketing",    "Content Writer",   "Part-Time", 48000.00, 1),
    ("EMP009", datetime(2021, 6, 14), "West",  "Finance",      "Financial Analyst","Full-Time", 85000.00, 4),
    ("EMP010", datetime(2019, 8, 20), "East",  "Engineering",  "DevOps Engineer",  "Full-Time", 105000.00, 7),
]

# Create DataFrame
employees_df = spark.createDataFrame(employees_data, schema=employees_schema)

# Show the data
employees_df.show(truncate=False)

# Print schema
employees_df.printSchema()

In [0]:
from pyspark.sql.functions import col, avg, sum, count, max, min, round, desc

# -----------------------------------------------
# 1. Average Salary by Department
# -----------------------------------------------
print("=== Average Salary by Department ===")
employees_df.groupBy("Department") \
    .agg(round(avg("Salary"), 2).alias("Avg_Salary")) \
    .orderBy(desc("Avg_Salary")) \
    .show()

# -----------------------------------------------
# 2. Headcount by Region
# -----------------------------------------------
print("=== Headcount by Region ===")
employees_df.groupBy("Region") \
    .agg(count("EmployeeID").alias("Total_Employees")) \
    .orderBy(desc("Total_Employees")) \
    .show()

# -----------------------------------------------
# 3. Total Salary Cost by Department and Job Category
# -----------------------------------------------
print("=== Total Salary Cost by Department & Category ===")
employees_df.groupBy("Department", "Category") \
    .agg(
        round(sum("Salary"), 2).alias("Total_Salary"),
        count("EmployeeID").alias("Headcount")
    ) \
    .orderBy("Department", "Category") \
    .show()

# -----------------------------------------------
# 4. Max and Min Salary by Region
# -----------------------------------------------
print("=== Max & Min Salary by Region ===")
employees_df.groupBy("Region") \
    .agg(
        max("Salary").alias("Max_Salary"),
        min("Salary").alias("Min_Salary"),
        round(avg("Salary"), 2).alias("Avg_Salary")
    ) \
    .orderBy("Region") \
    .show()

# -----------------------------------------------
# 5. Average Years of Experience by Job Title
# -----------------------------------------------
print("=== Avg Experience by Job Title ===")
employees_df.groupBy("JobTitle") \
    .agg(
        round(avg("YearsExperience"), 1).alias("Avg_Experience"),
        round(avg("Salary"), 2).alias("Avg_Salary")
    ) \
    .orderBy(desc("Avg_Experience")) \
    .show()

# -----------------------------------------------
# 6. Filter: High Earners (Salary > 90000)
# -----------------------------------------------
print("=== High Earners (Salary > 90,000) ===")
employees_df.filter(col("Salary") > 90000) \
    .select("EmployeeID", "JobTitle", "Department", "Salary") \
    .orderBy(desc("Salary")) \
    .show()

In [0]:
# Save as Delta Table (overwrite mode)
sales_transformed.write.format("delta").mode("overwrite").saveAsTable("one_data_uc.sales_schema.sales_git_table")